In [2]:
!pip install accelerate

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 23.8 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [accelerate]6 [accelerate]_hub]


In [ ]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 22.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 17.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]


In [ ]:
# Hugging Face: https://huggingface.co/microsoft/beit-base-patch16-224
# Paper: https://arxiv.org/pdf/2106.08254
from accelerate import Accelerator, ProfileKwargs
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BeitImageProcessor, BeitForImageClassification
import time
import os, time, socket, platform

batch_size = 2
model_name = "microsoft/beit-base-patch16-224"
total_examples = 32

# Load microsoft/swin-base-patch4-window7-224
processor = BeitImageProcessor.from_pretrained(model_name)
model = BeitForImageClassification.from_pretrained(model_name)

# ------- Save the trace ------------------
def trace_handler(profile):
    # Print profiling results
    print("Sort by cpu_time_total:")
    print(profile.key_averages().table(sort_by="cpu_time_total", row_limit=15))
    # Create directory if it does not exist
    os.makedirs("traces", exist_ok=True)
    profile.export_chrome_trace("traces/batch_size_" + str(batch_size) + ".json")

# -------- CPU RSS helpers (best effort) --------
def get_rss_bytes():
    """Current process RSS in bytes (best effort)."""
    try:
        import psutil  # type: ignore
        return psutil.Process(os.getpid()).memory_info().rss
    except Exception:
        pass
    try:
        import resource
        usage = resource.getrusage(resource.RUSAGE_SELF)
        # On Linux ru_maxrss is in KB, on macOS it is bytes.
        if platform.system().lower() == "linux":
            return int(usage.ru_maxrss * 1024)
        return int(usage.ru_maxrss)
    except Exception:
        return None

def format_bytes(n):
    if n is None:
        return "n/a"
    units = ["B", "KiB", "MiB", "GiB", "TiB"]
    x = float(n)
    for u in units:
        if x < 1024.0 or u == units[-1]:
            return f"{x:.2f} {u}"
        x /= 1024.0
    return f"{x:.2f} B"

# Create a batch of random images (3 color channels, 224x224 resolution)
all_input_images = torch.rand((total_examples, 3, 224, 224))  # Random image batch
all_labels = torch.randint(0, 2, (total_examples,))
dataset = TensorDataset(all_input_images, all_labels)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# Define profiling kwargs for CPU activities
profile_kwargs = ProfileKwargs(activities=["cpu"], profile_memory=True, record_shapes=True, with_flops=True, on_trace_ready=trace_handler)

# Initialize the accelerator for CPU
accelerator = Accelerator(cpu=True, kwargs_handlers=[profile_kwargs])

# Prepare the model for CPU execution
model, dataloader = accelerator.prepare(model, dataloader)
model.eval()  # Set model to evaluation mode

device = accelerator.device
rank = accelerator.process_index
world = accelerator.num_processes
local_rank = accelerator.local_process_index
host = socket.gethostname()

# ---- track memory baseline ----
cpu_rss_start = get_rss_bytes()
cpu_rss_peak = cpu_rss_start

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# Inference + profiling
total_seen_local = 0
t0 = time.time()


# Profile the model execution on the CPU
with accelerator.profile() as prof:
    with torch.no_grad():
        for inputs, _ in dataloader:
          inputs = inputs.to(device)
          outputs = model(inputs)  # Forward pass

          total_seen_local += inputs.size(0)
          # update cpu "peak" (best effort)
          rss = get_rss_bytes()
          if rss is not None and (cpu_rss_peak is None or rss > cpu_rss_peak):
              cpu_rss_peak = rss


accelerator.wait_for_everyone()
if torch.cuda.is_available():
    torch.cuda.synchronize()
t1 = time.time()

# Global processed (no gather_object needed)
global_seen = int(accelerator.reduce(torch.tensor(total_seen_local, device=device), reduction="sum").item())

local_throughput = total_seen_local / (t1 - t0) if (t1 - t0) > 0 else float("nan")
global_throughput = global_seen / (t1 - t0) if (t1 - t0) > 0 else float("nan")

# ---- final memory snapshot ----
cpu_rss_end = get_rss_bytes()


gpu_name = None
gpu_alloc = gpu_reserved = gpu_peak_alloc = gpu_peak_reserved = None
if torch.cuda.is_available():
    try:
        gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
    except Exception:
        gpu_name = None
    gpu_alloc = torch.cuda.memory_allocated()
    gpu_reserved = torch.cuda.memory_reserved()
    gpu_peak_alloc = torch.cuda.max_memory_allocated()
    gpu_peak_reserved = torch.cuda.max_memory_reserved()

if accelerator.is_main_process:
  print(
      f"\nDone. world_size={world} global_examples={global_seen} time={t1-t0:.2f}s "
      f"global_throughput={global_throughput:.1f} examples/s\n",
      flush=True
  )

# ---- Print each rank's profiler + memory (serialized) ----
accelerator.wait_for_everyone()
for r in range(world):
    if rank == r:
        print("=" * 70, flush=True)
        print(
            f"[RANK {rank}/{world}] host={host} local_rank={local_rank} device={device} gpu={gpu_name}\n"
            f"  local_examples={total_seen_local} local_throughput={local_throughput:.1f} examples/s\n"
            f"  CPU RSS: start={format_bytes(cpu_rss_start)}  end={format_bytes(cpu_rss_end)}  peak~={format_bytes(cpu_rss_peak)}\n"
            f"  GPU mem: alloc={format_bytes(gpu_alloc)}  reserved={format_bytes(gpu_reserved)}  "
            f"peak_alloc={format_bytes(gpu_peak_alloc)}  peak_reserved={format_bytes(gpu_peak_reserved)}",
            flush=True
        )
        print("=" * 70, flush=True)

    accelerator.wait_for_everyone()

config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/home/

Epoch 1/2, Loss: 0.6990
Epoch 2/2, Loss: 0.6940

Done. world_size=1 global_examples=64 time=5.77s global_throughput=11.1 examples/s

[RANK 0/1] host=ip-172-21-228-249.ec2.internal local_rank=0 device=cpu gpu=None
  local_examples=64 local_throughput=11.1 examples/s
  CPU RSS: start=802.02 MiB  end=2.04 GiB  peak~=2.04 GiB
  GPU mem: alloc=n/a  reserved=n/a  peak_alloc=n/a  peak_reserved=n/a
Sort by cpu_time_total:


ERROR:2026-05-21 21:56:53 10218:10218 DeviceProperties.cpp:47] gpuGetDeviceCount failed with code 100


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  Total MFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                               Optimizer.step#Adam.step         2.13%     113.376ms        43.41%        2.305s      72.038ms      33.46 MB      -1.05 GB            32            --  
                                             aten::sqrt        21.47%        1.140s        21.47%        1.140s     868.956us     535.42 MB     535.42 MB          1312            --  
                                             aten::gelu        11.54%     612.94

In [ ]:
import tempfile

def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        torch.save(model.state_dict(), tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 * 1024)
    os.remove(tmp.name)
    return size_mb

model_size = get_model_size_mb(model)
print(f"Model size: {model_size:.2f} MB")